# The Deceleration Engine: Quantifying Route Separation Through Combine Braking Dynamics
**NFL Big Data Bowl 2027 Analytics Track**

> *"Speed gets you noticed; deceleration gets you open."*

## 1. Executive Summary & Problem Formulation
Traditional NFL Scouting Combine evaluation has prioritized straight-line acceleration and 40-yard dash times. However, tracking audits show that while maximum speed translates from drills to games ( pprox 0.95$), static start and burst metrics carry shockingly low predictive power ( pprox 0.28$).

In the NFL, separation against sticky man coverage is created at the **inflection point** of route stems (Hitch, Out, In, Slant). In this study, we leverage 10Hz RFID sensor tracking data from both the **NFL Scouting Combine** and **in-game Next Gen Stats (NGS)** to prove that wide receivers who possess elite deceleration capabilities consistently generate wider throwing windows in the NFL.

### Key Contributions:
- **High-Order Kinematic Extraction:** Isolating tangential acceleration ($, braking) and normal acceleration ($, cornering) at 10Hz.
- **BRK+ (Braking Rate Plus):** A standardized, scout-friendly metric (calibrated to a 100 league baseline) identifying undervalued route technicians.
- **Empirical Validation:** Linking Combine route drills directly to real in-game target separation.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("..")
from src.data.loader import load_combine_metadata, generate_synthetic_route_play
from src.features.kinematics import smooth_tracking_data, compute_kinematics
from src.visualization.field import draw_nfl_field
from src.visualization.plots import plot_route_deceleration_overlay, plot_separation_breakdown

print("Libraries and project modules loaded.")

## 2. Combine Kinematic Extraction (3-Cone & Route Drills)
We calculate tangential acceleration  = rac{ec{v} \cdot ec{a}}{\|ec{v}\|}$ to measure pure braking force. Below are the processed results from the real Combine tracking dataset.

In [ ]:
master_df = pd.read_csv("../data/processed/master_deceleration_translation.csv")
top_brk = master_df.sort_values(by="BRK_plus", ascending=False).head(10)
cols = ["display_name", "combine_position", "forty", "BRK_plus", "in_game_mean_separation", "in_game_breaking_routes"]
top_brk[cols]

## 3. Empirical Translation: Combine BRK+ vs In-Game Separation
Does Combine deceleration translate to real NFL games? Below we plot player BRK+ against mean in-game separation at pass forward across breaking routes (Hitch, Out, In, Slant).

In [ ]:
plt.style.use("dark_background")
fig, ax = plt.subplots(figsize=(9, 6), facecolor="#0e1117")
ax.set_facecolor("#161b22")

sc = ax.scatter(
    master_df["BRK_plus"],
    master_df["in_game_mean_separation"],
    c=master_df["in_game_breaking_routes"],
    cmap="viridis",
    s=80,
    alpha=0.85,
    edgecolors="w",
    linewidth=0.5
)

m_slope, b_int = np.polyfit(master_df["BRK_plus"], master_df["in_game_mean_separation"], 1)
x_vals = np.linspace(master_df["BRK_plus"].min(), master_df["BRK_plus"].max(), 50)
ax.plot(x_vals, m_slope * x_vals + b_int, color="#58a6ff", linestyle="--", linewidth=2, label=f"Trend (Slope = +{m_slope:.3f})")

for _, row in top_brk.head(5).iterrows():
    ax.annotate(row["display_name"], (row["BRK_plus"], row["in_game_mean_separation"]), xytext=(6, 4), textcoords="offset points", fontsize=9, color="#f0f6fc", fontweight="bold")

ax.set_xlabel("Combine BRK+ (Braking Rate Plus, League Avg = 100)", fontsize=11, color="#c9d1d9")
ax.set_ylabel("In-Game Separation at Pass Forward (yards)", fontsize=11, color="#c9d1d9")
ax.set_title("Combine Braking (BRK+) vs In-Game Route Separation", fontsize=13, fontweight="bold", pad=12, color="#ffffff")
ax.grid(color="#30363d", linestyle="--", alpha=0.5)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("In-Game Breaking Routes Count", color="#c9d1d9")
plt.show()

## 4. Route Break Kinematic Reconstruction
We reconstruct 2D tracking trajectories on an authentic NFL field coordinate plane to illustrate the mechanical breakdown of defensive coverage.

In [ ]:
df_wr, df_db = generate_synthetic_route_play(route_type="comeback")
df_wr_kin = compute_kinematics(smooth_tracking_data(df_wr))
df_db_kin = compute_kinematics(smooth_tracking_data(df_db))

fig, ax = draw_nfl_field(xlim=(25, 55), ylim=(5, 25))
plot_route_deceleration_overlay(ax, df_wr_kin["x_smooth"], df_wr_kin["y_smooth"], df_wr_kin["calc_speed"], df_wr_kin["tangential_accel"], label="WR Route")
ax.plot(df_db_kin["x_smooth"], df_db_kin["y_smooth"], "--", color="#ffcc00", linewidth=1.8, label="DB Shadow")
ax.legend(loc="upper right", facecolor="#21262d", edgecolor="#30363d", labelcolor="#c9d1d9")
plt.show()

## 5. Scouting Implications & Recommendations
1. **Rethinking 40-Yard Dash Primacy:** While straight-line speed is essential for stretch vertical routes, it exhibits minimal correlation with separation on the route tree majority (Hitch, Out, In, Slant).
2. **Draft Exploitation:** Receivers with moderate 40 times (e.g. 4.55 - 4.65) but elite BRK+ (>120) represent massive draft value asymmetries.
3. **Actionable Training:** Coaches can use 10Hz deceleration profiles to train wide receivers on eccentric braking load capacity and minimize inflection contact injuries.